In [ ]:
using Distributed
using JLD2
using Plots

num_cores = length(Sys.cpu_info())
if nprocs()==1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end ;

@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using Dates
    using Distributions
end

In [ ]:
println("num_cores = $(num_cores)")

In [ ]:
type_lattice = "qubit_surface"

num_super_samples = 10
num_samples = Int(1e5)
Kmax = 200
Nv = 1 # Has to be 1 for qubit

dmin, dmax = 5, 5
drange = dmin : 2 : dmax

# ϵrange = (10.5 : 0.1 : 11.5)/100
ϵrange = [10.9, 11.0]./100

ϵdrange = []
for ϵ in ϵrange
    for d in drange
        push!(ϵdrange, [ϵ, d])
    end
end
println(ϵdrange)

num_samples_each_core = Int(ceil(num_samples/num_cores))
num_samples = Int(num_samples_each_core * num_cores);
num_total_samples = num_super_samples * num_samples
println([num_samples_each_core, num_samples, num_total_samples])



logfile = "$(type_lattice)_mwms_$(dmin)_$(dmax)_$(min(ϵrange...))_$(max(ϵrange...))_$(Kmax)_$(Nv)_$(num_total_samples)_log.txt"
    open(logfile, "w") do file
end

In [ ]:
p_mld_list2 = Dict(ϵdrange.=>[0.0 for _ in 1 : length(ϵdrange)])
t_mld_list2 = Dict(ϵdrange.=>[0.0 for _ in 1 : length(ϵdrange)])

Krange = 1 : Kmax
total_t = @elapsed for ind in 1 : num_super_samples  
    @time results = pmap(1:num_cores) do _        

        p_mld_list = Dict(ϵdrange.=>[[] for _ in 1 : length(ϵdrange)])
        t_mld_list = Dict(ϵdrange.=>[0.0 for _ in 1 : length(ϵdrange)])
        
        Ms = Dict()
        Ωs = Dict()
        Mperps = Dict()
        invtransposeMqs = Dict()
        for (ind_ϵd, ϵd) in enumerate(ϵdrange)
            ϵ, d = ϵd[1], Int(ϵd[2])
            σ = (2/π * log((1-ϵ)/ϵ))^(-1/2) # ϵ/(1-ϵ) = exp(-π/(2σ^2)) = exp(-(√π)^2/(2σ^2))
            
            if !(d in keys(Ms))
            
                M = surface_code_M(d) ; 
                Mperp = GKP_logical_operator_generator(M) 
                Ω = Ω_matrix(M)         
                invtransposeMq = inv(transpose(M))[1:2:end, 1:2:end]    
                Ms[d] = M
                Mperps[d] = Mperp
                Ωs[d] = Ω
                invtransposeMqs[d] = invtransposeMq
            end

            surface_code_z_logicals = surface_code_Z_logicals(d)
            surface_code_x_logicals = surface_code_X_logicals(d)
            surface_code_x_stabilizers = surface_code_X_stabilizers(d)
            surface_code_z_stabilizers = surface_code_Z_stabilizers(d)   
            
            p_mld = zeros(length(Krange)+1)
            t_mld = 0       

            ϵdtime = @elapsed for _ in 1 : num_samples_each_core
                ξ = √π * rand(Binomial(1, ϵ), 2d^2)
                ξ2 = √(2π) * Ms[d] * inv(Ωs[d]) * ξ
                s = ξ2 - floor.(ξ2/(2π)) * 2π
                ηs = -transpose(Ωs[d]*Mperps[d]) * s/√(2π) ; 
                b = inv(√(2π) * transpose(Mperps[d])) * (ηs-ξ)
                @assert norm(round.(Int, b) - b) < 1e-10   
                
                t_mld += @elapsed begin
                    ηsq = ηs[1:2:end]

                    ps_I, ps_X = LatticeAlgorithms.mwms_surface_square(ηsq, σ, Kmax; subspace="z", Nv=Nv)

                    ξq = ξ[1:2:end]                    
                    for k in 1 : Kmax+1
                        lstar = zeros(d^2)
                        p_I, p_X = ps_I[k], ps_X[k]
                        p_I > p_X ? (lstar) : (lstar[surface_code_x_logicals[1]] .= 1/√2 * √(2π))
                        recq = -ηsq + lstar
                        neterrorq = invtransposeMqs[d] * (recq+ξq) / √(2π)                
                        norm(round.(Int, neterrorq) - neterrorq) < 1e-10 ? nx = 0 : nx = 1
                        mod(nx, 2) == 0 ? (p_mld[k] += 1) : (p_mld[k] += 0)                            
                    end
                end                
            end
    
            p_mld_list[[ϵ, d]] = p_mld
            t_mld_list[[ϵ, d]] += t_mld
            
            if myid() == 2
                # Print the progress of the 2nd worker
                println(["$(ind)/$(num_super_samples), $(ind_ϵd)/$(length(ϵdrange)), $d, $(ϵdtime), $(string(now()))"])
                open(logfile, "a") do file
                    write(file, "$(ind)/$(num_super_samples), $(ind_ϵd)/$(length(ϵdrange)), $d, $(ϵdtime), $(string(now()))\n")
                end
            end
        end
        return p_mld_list, t_mld_list
    end ;
    p_mld_list  = merge(.+, [res[1] for res in results]...)
    t_mld_list  = merge(+, [res[2] for res in results]...)
    
    p_mld_list2 = merge(.+, p_mld_list2, p_mld_list)
    t_mld_list2 = merge(.+, t_mld_list2, t_mld_list)
    
end

println(total_t)
map!(x->x/num_total_samples, values(p_mld_list2))
map!(x->x/num_total_samples, values(t_mld_list2))

fn = "$(type_lattice)_mwms_$(dmin)_$(dmax)_$(min(ϵrange...))_$(max(ϵrange...))_$(Kmax)_$(Nv)_$(num_total_samples).jld2"
jldsave(fn; 
    ϵrange=ϵrange, 
    drange=drange, 
    num_samples=num_total_samples,
    p_list = p_mld_list2,
    t_list = t_mld_list2,        
)

# Compare to existing data

In [ ]:
function get_p0list_sorted(p_list, drange, ϵrange)
    p0list_sorted = sort(p_list)
    p0list_sorted = collect(values(p0list_sorted))
    p0list_sorted = reshape(p0list_sorted, (length(drange), length(ϵrange)))
    p0list_sorted = [p0list_sorted[:,i] for i in 1:size(p0list_sorted,2)]
    return p0list_sorted
end

In [ ]:
new_data = sort(load(fn))
new_p_list = new_data["p_list"]
new_p_list_sorted = get_p0list_sorted(new_p_list, drange, ϵrange)

In [ ]:
old_data = sort(load("data/qubit_surface_mwms_5_15_0.105_0.115_1000_1_1001744.jld2"))
old_p_list = Dict()
for (k, v) in old_data["p_list"]
    if k[2] ∈ drange && k[1] ∈ ϵrange
        old_p_list[k] = v
    end
end

old_p_list_sorted = get_p0list_sorted(old_p_list, drange, ϵrange)

In [ ]:
linecolors = get_color_palette(:auto, plot_color(:white))

gs = []
for (ind_ϵ, ϵ) in enumerate(ϵrange)
    g = plot()
    for (ind_d, d) in enumerate(drange)
        new_ps = new_p_list_sorted[ind_ϵ][ind_d]
        old_ps = old_p_list_sorted[ind_ϵ][ind_d]
        yerrnew = sqrt.(new_ps .* (1 .- new_ps) ./ num_total_samples)
        yerrold = sqrt.(old_ps .* (1 .- old_ps) ./ num_total_samples)
        plot!(new_ps, label="new data, d=$d, ϵ=$ϵ", marker=:circle, color=linecolors[ind_d], yerr=yerrnew)
        plot!(old_ps[1:2length(new_ps)], label="old data, d=$d, ϵ=$ϵ", marker=:star, color=linecolors[ind_d], yerr=yerrold)
    end
    plot!(xlabel="K", ylabel="fidelity", size=(1200, 400))
    push!(gs, g)
end

plot(gs..., layout=(length(gs), 1))

